# EZHAL — AI-Assisted Grid Resilience Prototype

This notebook contains the **final hackathon workflow** used for EZHAL.

**Pipeline:**  
Grid Simulation → Optimization → Supervised ML → Physics Validation → Safety Correction → Baseline Comparison → Live Demo

> **Prototype scope:** steady-state / quasi-static AC power-flow contingency study on the standard 9-bus `pandapower` case9 network. It is not an EMT or transient grid-forming controller.


## 1. Grid Model & Physical Assumptions

The prototype represents two renewable resources (solar and wind) with fixed inverter apparent-power ratings. Reactive-power capability is constrained by:

**P² + Q² ≤ S²**

The study uses a normal/restoration voltage criterion of **0.95–1.05 pu**.


In [ ]:
import copy
import numpy as np
import pandas as pd
import pandapower as pp

# Fixed inverter ratings
SOLAR_P_RATED = 163.0
WIND_P_RATED  = 85.0

SOLAR_S_FIXED = 1.10 * SOLAR_P_RATED   # 179.3 MVA
WIND_S_FIXED  = 1.10 * WIND_P_RATED    # 93.5 MVA

# Small representative sample
test_cases = [
    # Load, Solar, Wind, From Bus, To Bus
    (0.90, 0.60, 0.60, 7, 8),
    (1.00, 0.80, 0.80, 7, 8),
    (1.10, 1.00, 1.00, 7, 8),
    (1.00, 0.60, 1.00, 6, 7),
    (1.00, 1.00, 0.60, 8, 2),
    (1.20, 1.00, 1.00, 5, 6),
]

results = []

for load_level, solar_level, wind_level, from_bus, to_bus in test_cases:

    solar_p = SOLAR_P_RATED * solar_level
    wind_p  = WIND_P_RATED * wind_level

    # OLD assumption: S changes with renewable availability
    solar_S_old = 1.10 * solar_p
    wind_S_old  = 1.10 * wind_p

    solar_q_old = np.sqrt(max(solar_S_old**2 - solar_p**2, 0))
    wind_q_old  = np.sqrt(max(wind_S_old**2 - wind_p**2, 0))

    # CORRECTED assumption: fixed inverter nameplate S
    solar_q_fixed = np.sqrt(max(SOLAR_S_FIXED**2 - solar_p**2, 0))
    wind_q_fixed  = np.sqrt(max(WIND_S_FIXED**2 - wind_p**2, 0))

    results.append({
        "Load": load_level,
        "Solar_Level": solar_level,
        "Wind_Level": wind_level,
        "Line": f"{from_bus}-{to_bus}",

        "Solar_Q_Old": solar_q_old,
        "Solar_Q_Fixed": solar_q_fixed,
        "Solar_Q_Difference": solar_q_fixed - solar_q_old,

        "Wind_Q_Old": wind_q_old,
        "Wind_Q_Fixed": wind_q_fixed,
        "Wind_Q_Difference": wind_q_fixed - wind_q_old
    })

comparison = pd.DataFrame(results)

pd.set_option("display.max_columns", None)

print(comparison.round(2))

print("\nAverage extra Q capability with fixed rating:")
print(
    "Solar:",
    round(comparison["Solar_Q_Difference"].mean(), 2),
    "MVAr"
)

print(
    "Wind:",
    round(comparison["Wind_Q_Difference"].mean(), 2),
    "MVAr"
)

   Load  Solar_Level  Wind_Level Line  Solar_Q_Old  Solar_Q_Fixed  \
0   0.9          0.6         0.6  7-8        44.82         150.28   
1   1.0          0.8         0.8  7-8        59.76         123.06   
2   1.1          1.0         1.0  7-8        74.70          74.70   
3   1.0          0.6         1.0  6-7        44.82         150.28   
4   1.0          1.0         0.6  8-2        74.70          74.70   
5   1.2          1.0         1.0  5-6        74.70          74.70   

   Solar_Q_Difference  Wind_Q_Old  Wind_Q_Fixed  Wind_Q_Difference  
0              105.46       23.37         78.37              54.99  
1               63.31       31.16         64.17              33.01  
2                0.00       38.95         38.95               0.00  
3              105.46       38.95         38.95               0.00  
4                0.00       23.37         78.37              54.99  
5                0.00       38.95         38.95               0.00  

Average extra Q capability with 

## 2. Corrected Scenario Simulation
Generate the final targeted set of **875 operating/contingency scenarios**.


In [ ]:
import copy
import pandas as pd
import pandapower as pp

# ============================================================
# CORRECTED SIMULATION — 875 SCENARIOS
# ============================================================

target_loads = [0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
target_solar = [0.60, 0.70, 0.80, 0.90, 1.00]
target_wind  = [0.60, 0.70, 0.80, 0.90, 1.00]

target_lines = [
    (3, 6),
    (6, 7),
    (7, 8),
    (8, 2),
    (5, 6)
]

target_results = []

for load_level in target_loads:
    for solar_level in target_solar:
        for wind_level in target_wind:
            for from_bus, to_bus in target_lines:

                temp_net = copy.deepcopy(net)

                # Load condition
                temp_net.load["p_mw"] = net.load["p_mw"] * load_level
                temp_net.load["q_mvar"] = net.load["q_mvar"] * load_level

                # Renewable active power availability
                temp_net.gen.at[0, "p_mw"] = 163.0 * solar_level
                temp_net.gen.at[1, "p_mw"] = 85.0 * wind_level

                # Find contingency
                matching = temp_net.line[
                    (
                        (temp_net.line["from_bus"] + 1 == from_bus) &
                        (temp_net.line["to_bus"] + 1 == to_bus)
                    ) |
                    (
                        (temp_net.line["from_bus"] + 1 == to_bus) &
                        (temp_net.line["to_bus"] + 1 == from_bus)
                    )
                ].index

                status = "Unsolved"
                vmin = None
                vmax = None
                weakest_bus = None

                if len(matching) > 0:

                    temp_net.line.at[
                        matching[0], "in_service"
                    ] = False

                    try:
                        pp.runpp(
                            temp_net,
                            numba=False
                        )

                        status = "Solved"
                        vmin = float(temp_net.res_bus.vm_pu.min())
                        vmax = float(temp_net.res_bus.vm_pu.max())
                        weakest_bus = int(
                            temp_net.res_bus.vm_pu.idxmin() + 1
                        )

                    except:
                        pass

                target_results.append({
                    "Load_Level": load_level,
                    "Solar_Level": solar_level,
                    "Wind_Level": wind_level,
                    "From_Bus": from_bus,
                    "To_Bus": to_bus,
                    "Status": status,
                    "Min_Voltage": vmin,
                    "Max_Voltage": vmax,
                    "Weakest_Bus": weakest_bus
                })


target_df = pd.DataFrame(target_results)

# BOTH voltage limits
target_df["Security"] = target_df.apply(
    lambda row:
        "Secure"
        if (
            row["Status"] == "Solved"
            and row["Min_Voltage"] >= 0.95
            and row["Max_Voltage"] <= 1.05
        )
        else "Insecure",
    axis=1
)

insecure_target = target_df[
    target_df["Security"] == "Insecure"
].reset_index(drop=True)

# Save it so we NEVER have to regenerate it again
target_df.to_csv(
    "corrected_875_scenarios.csv",
    index=False
)

insecure_target.to_csv(
    "corrected_insecure_scenarios.csv",
    index=False
)

print("================================")
print("CORRECTED SIMULATION COMPLETE")
print("================================")

print("Total scenarios:", len(target_df))

print("\nStatus:")
print(target_df["Status"].value_counts())

print("\nSecurity:")
print(target_df["Security"].value_counts())

print("\nInsecure scenarios for optimizer:",
      len(insecure_target))

print("\nFiles saved successfully.")

CORRECTED SIMULATION COMPLETE
Total scenarios: 875

Status:
Status
Solved    875
Name: count, dtype: int64

Security:
Security
Insecure    659
Secure      216
Name: count, dtype: int64

Insecure scenarios for optimizer: 659

Files saved successfully.


## 3. Physics-Constrained Optimization
Find the minimum required support/headroom action for insecure scenarios.


In [ ]:
# ============================================================
# FAST CORRECTED OPTIMIZER V2
# Fixed Rating + Vmin/Vmax + Smart Fallback
# ============================================================

ZERO_TOL = 0.001

def fast_optimize_scenario(row):

    load_level  = float(row["Load_Level"])
    solar_level = float(row["Solar_Level"])
    wind_level  = float(row["Wind_Level"])
    from_bus    = int(row["From_Bus"])
    to_bus      = int(row["To_Bus"])

    solar_available = SOLAR_P_RATED * solar_level
    wind_available  = WIND_P_RATED * wind_level

    # --------------------------------------------------------
    # Build scenario once
    # --------------------------------------------------------
    scenario_net = copy.deepcopy(net)

    scenario_net.load["p_mw"] = net.load["p_mw"] * load_level
    scenario_net.load["q_mvar"] = net.load["q_mvar"] * load_level

    matching = scenario_net.line[
        (
            (scenario_net.line["from_bus"] + 1 == from_bus) &
            (scenario_net.line["to_bus"] + 1 == to_bus)
        ) |
        (
            (scenario_net.line["from_bus"] + 1 == to_bus) &
            (scenario_net.line["to_bus"] + 1 == from_bus)
        )
    ].index

    if len(matching) == 0:
        return {
            "Status": "Contingency Not Found",
            "Action_Type": "Escalate",
            "PowerFlow_Evaluations": 0
        }

    scenario_net.line.at[matching[0], "in_service"] = False

    cache = {}

    # ========================================================
    # PHYSICS EVALUATION
    # ========================================================
    def evaluate(x):

        solar_h, wind_h, solar_v, wind_v = map(float, x)

        key = tuple(np.round(
            [solar_h, wind_h, solar_v, wind_v], 8
        ))

        if key in cache:
            return cache[key]

        n = copy.deepcopy(scenario_net)

        solar_p = max(solar_available - solar_h, 0.0)
        wind_p  = max(wind_available - wind_h, 0.0)

        n.gen.at[0, "p_mw"] = solar_p
        n.gen.at[1, "p_mw"] = wind_p

        n.gen.at[0, "vm_pu"] = solar_v
        n.gen.at[1, "vm_pu"] = wind_v

        # FIXED inverter ratings
        solar_qmax = np.sqrt(
            max(SOLAR_S_RATED**2 - solar_p**2, 0.0)
        )

        wind_qmax = np.sqrt(
            max(WIND_S_RATED**2 - wind_p**2, 0.0)
        )

        n.gen.at[0, "max_q_mvar"] = solar_qmax
        n.gen.at[0, "min_q_mvar"] = -solar_qmax

        n.gen.at[1, "max_q_mvar"] = wind_qmax
        n.gen.at[1, "min_q_mvar"] = -wind_qmax

        try:

            pp.runpp(
                n,
                enforce_q_lims=True,
                numba=False
            )

            result = {
                "solved": True,

                "vmin": float(n.res_bus.vm_pu.min()),
                "vmax": float(n.res_bus.vm_pu.max()),

                "weakest_bus":
                    int(n.res_bus.vm_pu.idxmin() + 1),

                "solar_q":
                    float(n.res_gen.at[0, "q_mvar"]),

                "wind_q":
                    float(n.res_gen.at[1, "q_mvar"]),

                "solar_qmax": solar_qmax,
                "wind_qmax": wind_qmax
            }

        except Exception:

            result = {"solved": False}

        cache[key] = result

        return result

    # ========================================================
    # HELPER: IS THIS ACTION SECURE?
    # ========================================================
    def is_secure(r):

        return (
            r["solved"] and
            r["vmin"] >= V_MIN - 1e-5 and
            r["vmax"] <= V_MAX + 1e-5
        )

    # ========================================================
    # BASE STATE FEATURES — BEFORE ACTION
    # ========================================================
    base = evaluate([
        0.0,
        0.0,
        1.00,
        1.00
    ])

    if base["solved"]:
        base_vmin = base["vmin"]
        base_vmax = base["vmax"]
        base_weakest = base["weakest_bus"]

        base_solar_q = base["solar_q"]
        base_wind_q  = base["wind_q"]

        base_solar_qmax = base["solar_qmax"]
        base_wind_qmax  = base["wind_qmax"]

    else:
        base_vmin = np.nan
        base_vmax = np.nan
        base_weakest = np.nan

        base_solar_q = np.nan
        base_wind_q = np.nan

        base_solar_qmax = np.nan
        base_wind_qmax = np.nan

    # ========================================================
    # RESULT BUILDER
    # ========================================================
    def build_result(x, final, status="Feasible"):

        solar_h = max(float(x[0]), 0.0)
        wind_h  = max(float(x[1]), 0.0)

        # Remove numerical noise
        if solar_h < ZERO_TOL:
            solar_h = 0.0

        if wind_h < ZERO_TOL:
            wind_h = 0.0

        total_curtailment = solar_h + wind_h

        if total_curtailment <= ZERO_TOL:
            action_type = "Support Only"
        else:
            action_type = "Headroom + Support"

        return {
            # ---------------- GRID INPUT ----------------
            "Load_Level": load_level,
            "Solar_Level": solar_level,
            "Wind_Level": wind_level,

            "Solar_Available_MW": solar_available,
            "Wind_Available_MW": wind_available,

            "From_Bus": from_bus,
            "To_Bus": to_bus,

            # ---------------- BASE FEATURES -------------
            "Base_Min_Voltage": base_vmin,
            "Base_Max_Voltage": base_vmax,
            "Weakest_Bus_Before": base_weakest,

            "Base_Solar_Q_MVAr": base_solar_q,
            "Base_Wind_Q_MVAr": base_wind_q,

            "Base_Solar_Q_Capability_MVAr":
                base_solar_qmax,

            "Base_Wind_Q_Capability_MVAr":
                base_wind_qmax,

            "Base_Solar_Q_Margin_MVAr":
                (
                    base_solar_qmax - abs(base_solar_q)
                    if not np.isnan(base_solar_q)
                    else np.nan
                ),

            "Base_Wind_Q_Margin_MVAr":
                (
                    base_wind_qmax - abs(base_wind_q)
                    if not np.isnan(base_wind_q)
                    else np.nan
                ),

            # ---------------- AI LABELS -----------------
            "Solar_Headroom_MW": solar_h,
            "Wind_Headroom_MW": wind_h,

            "Solar_Support_pu": float(x[2]),
            "Wind_Support_pu": float(x[3]),

            # ---------------- FINAL STATE ---------------
            "Final_Min_Voltage": final["vmin"],
            "Final_Max_Voltage": final["vmax"],
            "Weakest_Bus_After": final["weakest_bus"],

            "Solar_Q_MVAr": final["solar_q"],
            "Wind_Q_MVAr": final["wind_q"],

            "Solar_Q_Capability_MVAr":
                final["solar_qmax"],

            "Wind_Q_Capability_MVAr":
                final["wind_qmax"],

            "Total_Curtailment_MW":
                total_curtailment,

            "Total_Support_Increase_pu":
                (
                    float(x[2]) - 1.0 +
                    float(x[3]) - 1.0
                ),

            "Action_Type": action_type,
            "Status": status,

            "PowerFlow_Evaluations": len(cache)
        }

    # ========================================================
    # STEP 1 — ZERO HEADROOM CHECK
    # ========================================================

    max_support = evaluate([
        0.0,
        0.0,
        1.05,
        1.05
    ])

    if is_secure(max_support):

        def support_objective(v):
            return (
                (v[0] - 1.0) +
                (v[1] - 1.0)
            )

        def low_support(v):
            r = evaluate([
                0.0, 0.0, v[0], v[1]
            ])

            return (
                r["vmin"] - V_MIN
                if r["solved"]
                else -1.0
            )

        def high_support(v):
            r = evaluate([
                0.0, 0.0, v[0], v[1]
            ])

            return (
                V_MAX - r["vmax"]
                if r["solved"]
                else -1.0
            )

        # First fast attempt
        support_starts = [
            [1.025, 1.025]
        ]

        # Fallback only if necessary
        support_starts += [
            [1.05, 1.00],
            [1.00, 1.05],
            [1.05, 1.05]
        ]

        support_candidates = []

        for start in support_starts:

            opt = minimize(
                support_objective,
                start,
                method="SLSQP",

                bounds=[
                    (1.00, 1.05),
                    (1.00, 1.05)
                ],

                constraints=[
                    {
                        "type": "ineq",
                        "fun": low_support
                    },
                    {
                        "type": "ineq",
                        "fun": high_support
                    }
                ],

                options={
                    "maxiter": 40,
                    "ftol": 1e-7
                }
            )

            x = [
                0.0,
                0.0,
                float(opt.x[0]),
                float(opt.x[1])
            ]

            final = evaluate(x)

            if is_secure(final):

                support_candidates.append(
                    (x, final)
                )

                # Fast exit if first attempt worked
                if start == [1.025, 1.025]:
                    break

        if len(support_candidates) > 0:

            best_x, best_final = min(
                support_candidates,
                key=lambda z:
                    (
                        z[0][2] - 1.0
                        +
                        z[0][3] - 1.0
                    )
            )

            return build_result(
                best_x,
                best_final
            )

    # ========================================================
    # STEP 2 — HEADROOM REQUIRED
    # ========================================================

    def curtailment_objective(x):
        return x[0] + x[1]

    def low_constraint(x):

        r = evaluate(x)

        return (
            r["vmin"] - V_MIN
            if r["solved"]
            else -1.0
        )

    def high_constraint(x):

        r = evaluate(x)

        return (
            V_MAX - r["vmax"]
            if r["solved"]
            else -1.0
        )

    bounds = [
        (0.0, solar_available),
        (0.0, wind_available),
        (1.00, 1.05),
        (1.00, 1.05)
    ]

    # Fast start + fallback starts
    full_starts = [
        [
            0.02 * solar_available,
            0.02 * wind_available,
            1.05,
            1.05
        ],
        [
            0.0,
            0.0,
            1.05,
            1.05
        ],
        [
            0.05 * solar_available,
            0.05 * wind_available,
            1.05,
            1.05
        ],
        [
            0.10 * solar_available,
            0.10 * wind_available,
            1.05,
            1.05
        ]
    ]

    candidates = []

    for start in full_starts:

        opt = minimize(
            curtailment_objective,
            start,
            method="SLSQP",

            bounds=bounds,

            constraints=[
                {
                    "type": "ineq",
                    "fun": low_constraint
                },
                {
                    "type": "ineq",
                    "fun": high_constraint
                }
            ],

            options={
                "maxiter": 70,
                "ftol": 1e-7
            }
        )

        final = evaluate(opt.x)

        if is_secure(final):
            candidates.append(
                (opt.x.copy(), final)
            )

    # ========================================================
    # STEP 3 — TRUE ESCALATE AFTER FALLBACKS
    # ========================================================

    if len(candidates) == 0:

        return {
            "Load_Level": load_level,
            "Solar_Level": solar_level,
            "Wind_Level": wind_level,

            "Solar_Available_MW": solar_available,
            "Wind_Available_MW": wind_available,

            "From_Bus": from_bus,
            "To_Bus": to_bus,

            "Base_Min_Voltage": base_vmin,
            "Base_Max_Voltage": base_vmax,
            "Weakest_Bus_Before": base_weakest,

            "Base_Solar_Q_MVAr": base_solar_q,
            "Base_Wind_Q_MVAr": base_wind_q,

            "Base_Solar_Q_Capability_MVAr":
                base_solar_qmax,

            "Base_Wind_Q_Capability_MVAr":
                base_wind_qmax,

            "Base_Solar_Q_Margin_MVAr":
                (
                    base_solar_qmax - abs(base_solar_q)
                    if not np.isnan(base_solar_q)
                    else np.nan
                ),

            "Base_Wind_Q_Margin_MVAr":
                (
                    base_wind_qmax - abs(base_wind_q)
                    if not np.isnan(base_wind_q)
                    else np.nan
                ),

            "Solar_Headroom_MW": np.nan,
            "Wind_Headroom_MW": np.nan,

            "Solar_Support_pu": np.nan,
            "Wind_Support_pu": np.nan,

            "Final_Min_Voltage": np.nan,
            "Final_Max_Voltage": np.nan,

            "Total_Curtailment_MW": np.nan,

            "Action_Type": "Escalate",
            "Status": "No Feasible Solution",

            "PowerFlow_Evaluations": len(cache)
        }

    # ========================================================
    # STEP 4 — PICK LOWEST CURTAILMENT
    # ========================================================

    best_x, best_final = min(
        candidates,
        key=lambda z:
            float(z[0][0]) + float(z[0][1])
    )

    return build_result(
        best_x,
        best_final
    )


print("Fast Corrected Optimizer V2 loaded ✅")

Fast Corrected Optimizer V2 loaded ✅


In [ ]:
# ============================================================
# RUN FINAL V2 OPTIMIZER ON ALL 659 INSECURE SCENARIOS
# ============================================================

import os
import time
import pandas as pd

checkpoint_file = "final_v2_optimizer_checkpoint.csv"
final_file = "final_v2_optimizer_dataset.csv"

# Start fresh for THIS V2 run
all_results = []

start_time = time.time()

for i, (_, row) in enumerate(
    insecure_target.iterrows(),
    start=1
):

    result = fast_optimize_scenario(row)
    all_results.append(result)

    # Save every 25 scenarios
    if i % 25 == 0:

        pd.DataFrame(all_results).to_csv(
            checkpoint_file,
            index=False
        )

        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = avg_time * (len(insecure_target) - i)

        print(
            f"Completed {i}/{len(insecure_target)} | "
            f"Elapsed: {elapsed/60:.1f} min | "
            f"Estimated remaining: {remaining/60:.1f} min"
        )

# Final save
final_v2_df = pd.DataFrame(all_results)

final_v2_df.to_csv(
    final_file,
    index=False
)

elapsed = time.time() - start_time

print("\n====================================")
print("FINAL V2 OPTIMIZATION COMPLETE")
print("====================================")

print("Total scenarios:", len(final_v2_df))
print("Total time:", round(elapsed / 60, 2), "minutes")

print("\nSTATUS:")
print(final_v2_df["Status"].value_counts())

print("\nACTION TYPES:")
print(final_v2_df["Action_Type"].value_counts(dropna=False))

feasible = final_v2_df[
    final_v2_df["Status"] == "Feasible"
].copy()

print("\nFeasible scenarios:", len(feasible))

print(
    "Escalated scenarios:",
    (final_v2_df["Status"] != "Feasible").sum()
)

if len(feasible) > 0:

    print("\n--- HEADROOM ---")

    print(
        "Zero-headroom cases:",
        (feasible["Total_Curtailment_MW"] <= ZERO_TOL).sum()
    )

    print(
        "Headroom-required cases:",
        (feasible["Total_Curtailment_MW"] > ZERO_TOL).sum()
    )

    print(
        "Average curtailment:",
        round(feasible["Total_Curtailment_MW"].mean(), 4),
        "MW"
    )

    print(
        "Maximum curtailment:",
        round(feasible["Total_Curtailment_MW"].max(), 4),
        "MW"
    )

    print("\n--- VOLTAGE VALIDATION ---")

    print(
        "Minimum final voltage:",
        round(feasible["Final_Min_Voltage"].min(), 6)
    )

    print(
        "Maximum final voltage:",
        round(feasible["Final_Max_Voltage"].max(), 6)
    )

    physics_pass = (
        (feasible["Final_Min_Voltage"] >= V_MIN - 1e-5)
        &
        (feasible["Final_Max_Voltage"] <= V_MAX + 1e-5)
    )

    print(
        "Physics-valid feasible cases:",
        f"{physics_pass.sum()}/{len(feasible)}"
    )

    print(
        "Physics validation rate:",
        round(physics_pass.mean() * 100, 2),
        "%"
    )

print("\nSaved as:", final_file)

Completed 25/659 | Elapsed: 0.5 min | Estimated remaining: 13.5 min
Completed 50/659 | Elapsed: 1.8 min | Estimated remaining: 22.0 min
Completed 75/659 | Elapsed: 4.2 min | Estimated remaining: 33.0 min
Completed 100/659 | Elapsed: 6.5 min | Estimated remaining: 36.5 min
Completed 125/659 | Elapsed: 7.4 min | Estimated remaining: 31.7 min
Completed 150/659 | Elapsed: 8.7 min | Estimated remaining: 29.7 min
Completed 175/659 | Elapsed: 10.0 min | Estimated remaining: 27.7 min
Completed 200/659 | Elapsed: 11.4 min | Estimated remaining: 26.1 min
Completed 225/659 | Elapsed: 12.3 min | Estimated remaining: 23.8 min
Completed 250/659 | Elapsed: 13.8 min | Estimated remaining: 22.5 min
Completed 275/659 | Elapsed: 14.8 min | Estimated remaining: 20.6 min
Completed 300/659 | Elapsed: 16.4 min | Estimated remaining: 19.6 min
Completed 325/659 | Elapsed: 17.6 min | Estimated remaining: 18.0 min
Completed 350/659 | Elapsed: 19.1 min | Estimated remaining: 16.9 min
Completed 375/659 | Elapsed: 

## 4. Machine Learning Dataset
Use only **pre-action grid features** to avoid data leakage.


In [ ]:
# ============================================================
# ML STEP 1 — LOAD & INSPECT FINAL DATASET
# ============================================================

import pandas as pd
import numpy as np

ml_df = pd.read_csv("final_v2_optimizer_dataset.csv")

print("Dataset shape:", ml_df.shape)

print("\nStatus:")
print(ml_df["Status"].value_counts())

print("\nAction Type:")
print(ml_df["Action_Type"].value_counts())

print("\nMissing values:")
missing = ml_df.isna().sum()
print(missing[missing > 0])

print("\nColumns:")
for i, col in enumerate(ml_df.columns, 1):
    print(i, col)

Dataset shape: (659, 32)

Status:
Status
Feasible                434
No Feasible Solution    225
Name: count, dtype: int64

Action Type:
Action_Type
Support Only          378
Escalate              225
Headroom + Support     56
Name: count, dtype: int64

Missing values:
Base_Min_Voltage                  5
Base_Max_Voltage                  5
Weakest_Bus_Before                5
Base_Solar_Q_MVAr                 5
Base_Wind_Q_MVAr                  5
Base_Solar_Q_Capability_MVAr      5
Base_Wind_Q_Capability_MVAr       5
Base_Solar_Q_Margin_MVAr          5
Base_Wind_Q_Margin_MVAr           5
Solar_Headroom_MW               225
Wind_Headroom_MW                225
Solar_Support_pu                225
Wind_Support_pu                 225
Final_Min_Voltage               225
Final_Max_Voltage               225
Weakest_Bus_After               225
Solar_Q_MVAr                    225
Wind_Q_MVAr                     225
Solar_Q_Capability_MVAr         225
Wind_Q_Capability_MVAr          225
Total_Curt

In [ ]:
# ============================================================
# ML STEP 2 — PREPARE FEATURES & TARGET
# NO DATA LEAKAGE
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

# Features available BEFORE the AI makes its decision
feature_columns = [
    "Load_Level",
    "Solar_Level",
    "Wind_Level",

    "Solar_Available_MW",
    "Wind_Available_MW",

    "From_Bus",
    "To_Bus",

    "Base_Min_Voltage",
    "Base_Max_Voltage",
    "Weakest_Bus_Before",

    "Base_Solar_Q_MVAr",
    "Base_Wind_Q_MVAr",

    "Base_Solar_Q_Capability_MVAr",
    "Base_Wind_Q_Capability_MVAr",

    "Base_Solar_Q_Margin_MVAr",
    "Base_Wind_Q_Margin_MVAr"
]

X = ml_df[feature_columns].copy()
y = ml_df["Action_Type"].copy()

# Only 5 rows have missing BASE-state information.
# Do not invent physical measurements for them.
valid_rows = X.notna().all(axis=1)

X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

ml_clean = ml_df.loc[valid_rows].reset_index(drop=True)

print("Original rows:", len(ml_df))
print("Rows removed because base physics failed:",
      (~valid_rows).sum())
print("Rows available for ML:", len(X))

print("\nTarget distribution:")
print(y.value_counts())

# ------------------------------------------------------------
# GROUPED TRAIN / TEST SPLIT
#
# Same operating condition must NOT appear in both sets.
# This is stronger than a simple random split.
# ------------------------------------------------------------

groups = (
    ml_clean["Load_Level"].astype(str) + "_" +
    ml_clean["Solar_Level"].astype(str) + "_" +
    ml_clean["Wind_Level"].astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test  = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test  = y.iloc[test_idx].copy()

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTrain classes:")
print(y_train.value_counts())

print("\nTest classes:")
print(y_test.value_counts())

print("\nNumber of input features:", X.shape[1])

print(
    "\nOperating-state overlap:",
    len(
        set(groups.iloc[train_idx])
        &
        set(groups.iloc[test_idx])
    )
)

Original rows: 659
Rows removed because base physics failed: 5
Rows available for ML: 654

Target distribution:
Action_Type
Support Only          378
Escalate              225
Headroom + Support     51
Name: count, dtype: int64

Train rows: 528
Test rows: 126

Train classes:
Action_Type
Support Only          304
Escalate              181
Headroom + Support     43
Name: count, dtype: int64

Test classes:
Action_Type
Support Only          74
Escalate              44
Headroom + Support     8
Name: count, dtype: int64

Number of input features: 16

Operating-state overlap: 0


## 5. Action Classification
Extra Trees classifier predicts the required action type.


In [ ]:
# ============================================================
# ML STEP 3 — TRAIN & COMPARE CLASSIFICATION MODELS
# ============================================================

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    )
}

results = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    balanced_acc = balanced_accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, average="macro")

    results[name] = {
        "model": model,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1
    }

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Accuracy:", round(accuracy, 4))
    print("Balanced Accuracy:", round(balanced_acc, 4))
    print("Macro F1:", round(macro_f1, 4))

    print("\nClassification Report:")
    print(classification_report(y_test, pred, digits=4))

    print("Confusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            pred,
            labels=[
                "Support Only",
                "Headroom + Support",
                "Escalate"
            ]
        )
    )

# Select model using Macro F1
best_name = max(
    results,
    key=lambda name: results[name]["macro_f1"]
)

best_classifier = results[best_name]["model"]

print("\n" + "=" * 60)
print("BEST CLASSIFIER:", best_name)
print("Best Macro F1:",
      round(results[best_name]["macro_f1"], 4))
print("=" * 60)


Random Forest
Accuracy: 0.9762
Balanced Accuracy: 0.9865
Macro F1: 0.9405

Classification Report:
                    precision    recall  f1-score   support

          Escalate     1.0000    1.0000    1.0000        44
Headroom + Support     0.7273    1.0000    0.8421         8
      Support Only     1.0000    0.9595    0.9793        74

          accuracy                         0.9762       126
         macro avg     0.9091    0.9865    0.9405       126
      weighted avg     0.9827    0.9762    0.9778       126

Confusion Matrix:
[[71  3  0]
 [ 0  8  0]
 [ 0  0 44]]

Extra Trees
Accuracy: 0.9762
Balanced Accuracy: 0.9865
Macro F1: 0.9523

Classification Report:
                    precision    recall  f1-score   support

          Escalate     0.9778    1.0000    0.9888        44
Headroom + Support     0.8000    1.0000    0.8889         8
      Support Only     1.0000    0.9595    0.9793        74

          accuracy                         0.9762       126
         macro avg     0

## 6. Where + How Much Regression
Extra Trees regressor predicts solar/wind headroom and voltage-support setpoints.


In [ ]:
# ============================================================
# ML STEP 4 — PREPARE MULTI-OUTPUT REGRESSION DATA
# Predict WHERE + HOW MUCH
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

# Keep only optimizer-feasible cases
reg_df = ml_clean[
    ml_clean["Status"] == "Feasible"
].copy()

reg_features = feature_columns

reg_targets = [
    "Solar_Headroom_MW",
    "Wind_Headroom_MW",
    "Solar_Support_pu",
    "Wind_Support_pu"
]

X_reg = reg_df[reg_features].copy()
y_reg = reg_df[reg_targets].copy()

# Final safety check
valid_reg = (
    X_reg.notna().all(axis=1) &
    y_reg.notna().all(axis=1)
)

X_reg = X_reg.loc[valid_reg].reset_index(drop=True)
y_reg = y_reg.loc[valid_reg].reset_index(drop=True)
reg_df = reg_df.loc[valid_reg].reset_index(drop=True)

# Group by operating state to prevent leakage
reg_groups = (
    reg_df["Load_Level"].astype(str) + "_" +
    reg_df["Solar_Level"].astype(str) + "_" +
    reg_df["Wind_Level"].astype(str)
)

reg_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

reg_train_idx, reg_test_idx = next(
    reg_splitter.split(
        X_reg,
        y_reg,
        groups=reg_groups
    )
)

X_reg_train = X_reg.iloc[reg_train_idx].copy()
X_reg_test  = X_reg.iloc[reg_test_idx].copy()

y_reg_train = y_reg.iloc[reg_train_idx].copy()
y_reg_test  = y_reg.iloc[reg_test_idx].copy()

print("Regression dataset:", len(X_reg))
print("Train:", len(X_reg_train))
print("Test:", len(X_reg_test))

print("\nTarget summary:")
print(y_reg.describe().T[
    ["mean", "std", "min", "max"]
])

print(
    "\nOperating-state overlap:",
    len(
        set(reg_groups.iloc[reg_train_idx]) &
        set(reg_groups.iloc[reg_test_idx])
    )
)

print("\nHeadroom cases in train:",
      ((y_reg_train["Solar_Headroom_MW"] +
        y_reg_train["Wind_Headroom_MW"]) > 0.001).sum())

print("Headroom cases in test:",
      ((y_reg_test["Solar_Headroom_MW"] +
        y_reg_test["Wind_Headroom_MW"]) > 0.001).sum())

Regression dataset: 429
Train: 347
Test: 82

Target summary:
                       mean       std  min        max
Solar_Headroom_MW  0.671724  3.596581  0.0  39.250036
Wind_Headroom_MW   1.268641  4.950087  0.0  37.996639
Solar_Support_pu   1.023153  0.018936  1.0   1.050000
Wind_Support_pu    1.018624  0.019777  1.0   1.050000

Operating-state overlap: 0

Headroom cases in train: 43
Headroom cases in test: 8


In [ ]:
# ============================================================
# ML STEP 5 — TRAIN & COMPARE MULTI-OUTPUT REGRESSORS
# ============================================================

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

reg_models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    )
}

reg_results = {}

for name, model in reg_models.items():

    model.fit(X_reg_train, y_reg_train)
    pred = model.predict(X_reg_test)

    pred_df = pd.DataFrame(
        pred,
        columns=reg_targets,
        index=y_reg_test.index
    )

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    target_results = []

    for target in reg_targets:

        true = y_reg_test[target]
        predicted = pred_df[target]

        mae = mean_absolute_error(true, predicted)
        rmse = np.sqrt(mean_squared_error(true, predicted))
        r2 = r2_score(true, predicted)

        target_results.append({
            "Target": target,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    target_results = pd.DataFrame(target_results)

    print(target_results.to_string(index=False))

    average_mae = target_results["MAE"].mean()
    average_r2 = target_results["R2"].mean()

    reg_results[name] = {
        "model": model,
        "predictions": pred_df,
        "metrics": target_results,
        "average_mae": average_mae,
        "average_r2": average_r2
    }

    print("\nAverage R2:", round(average_r2, 4))


# ------------------------------------------------------------
# Compare HEADROOM cases separately
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("HEADROOM-REQUIRED TEST CASES ONLY")
print("=" * 65)

headroom_test_mask = (
    y_reg_test["Solar_Headroom_MW"] +
    y_reg_test["Wind_Headroom_MW"]
) > 0.001

print("Number of headroom test cases:",
      headroom_test_mask.sum())

for name in reg_results:

    pred_df = reg_results[name]["predictions"]

    print("\n", name)

    for target in [
        "Solar_Headroom_MW",
        "Wind_Headroom_MW"
    ]:

        true = y_reg_test.loc[
            headroom_test_mask, target
        ]

        predicted = pred_df.loc[
            headroom_test_mask, target
        ]

        mae = mean_absolute_error(
            true,
            predicted
        )

        print(
            target,
            "Headroom-case MAE:",
            round(mae, 4),
            "MW"
        )


# Choose based on overall target R2 for now
best_reg_name = max(
    reg_results,
    key=lambda name: reg_results[name]["average_r2"]
)

best_regressor = reg_results[best_reg_name]["model"]

print("\n" + "=" * 65)
print("CURRENT BEST REGRESSOR:", best_reg_name)
print("=" * 65)


Random Forest
           Target      MAE     RMSE       R2
Solar_Headroom_MW 0.358830 1.403958 0.657579
 Wind_Headroom_MW 0.528481 2.184903 0.572202
 Solar_Support_pu 0.001119 0.002024 0.986804
  Wind_Support_pu 0.001157 0.002429 0.985012

Average R2: 0.8004

Extra Trees
           Target      MAE     RMSE       R2
Solar_Headroom_MW 0.277089 1.150878 0.769903
 Wind_Headroom_MW 0.397965 1.677100 0.747947
 Solar_Support_pu 0.000957 0.002154 0.985057
  Wind_Support_pu 0.000648 0.001724 0.992448

Average R2: 0.8738

HEADROOM-REQUIRED TEST CASES ONLY
Number of headroom test cases: 8

 Random Forest
Solar_Headroom_MW Headroom-case MAE: 2.5606 MW
Wind_Headroom_MW Headroom-case MAE: 4.1217 MW

 Extra Trees
Solar_Headroom_MW Headroom-case MAE: 2.1231 MW
Wind_Headroom_MW Headroom-case MAE: 2.9799 MW

CURRENT BEST REGRESSOR: Extra Trees


In [ ]:
# ============================================================
# ML STEP 6 — SAVE SELECTED MODELS
# ============================================================

import joblib

joblib.dump(
    best_classifier,
    "final_action_classifier.pkl"
)

joblib.dump(
    best_regressor,
    "final_action_regressor.pkl"
)

print("Saved classifier: Extra Trees")
print("Saved regressor: Extra Trees")

print("\nClassifier Macro F1:",
      round(results["Extra Trees"]["macro_f1"], 4))

print("Regressor Average R2:",
      round(reg_results["Extra Trees"]["average_r2"], 4))

print("\nModels saved successfully.")

Saved classifier: Extra Trees
Saved regressor: Extra Trees

Classifier Macro F1: 0.9523
Regressor Average R2: 0.8738

Models saved successfully.


## 7. Physics-in-the-Loop Validation
Apply unseen ML decisions back to the power-flow model.


In [ ]:
# ============================================================
# ML STEP 7 — PHYSICS-IN-THE-LOOP VALIDATION
# AI ACTION -> PANDAPOWER -> SECURITY CHECK
# ============================================================

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn

V_MIN = 0.95
V_MAX = 1.05

SOLAR_P_RATED = 163.0
WIND_P_RATED = 85.0

SOLAR_S_RATED = 1.10 * SOLAR_P_RATED
WIND_S_RATED = 1.10 * WIND_P_RATED

# Extra Trees predictions
ai_predictions = best_regressor.predict(X_reg_test)

ai_pred_df = pd.DataFrame(
    ai_predictions,
    columns=reg_targets,
    index=X_reg_test.index
)

validation_results = []

for idx in X_reg_test.index:

    row = reg_df.loc[idx]
    pred = ai_pred_df.loc[idx]

    # -----------------------------
    # 1. AI predicted actions
    # -----------------------------

    solar_hr = max(
        0.0,
        min(
            float(pred["Solar_Headroom_MW"]),
            float(row["Solar_Available_MW"])
        )
    )

    wind_hr = max(
        0.0,
        min(
            float(pred["Wind_Headroom_MW"]),
            float(row["Wind_Available_MW"])
        )
    )

    solar_support = np.clip(
        float(pred["Solar_Support_pu"]),
        1.00,
        1.05
    )

    wind_support = np.clip(
        float(pred["Wind_Support_pu"]),
        1.00,
        1.05
    )

    # -----------------------------
    # 2. Build fresh grid
    # -----------------------------

    test_net = pn.case9()

    # Scale loads
    test_net.load["p_mw"] *= float(row["Load_Level"])
    test_net.load["q_mvar"] *= float(row["Load_Level"])

    # Renewable available power minus AI headroom
    solar_p = max(
        0.0,
        float(row["Solar_Available_MW"]) - solar_hr
    )

    wind_p = max(
        0.0,
        float(row["Wind_Available_MW"]) - wind_hr
    )

    # case9:
    # gen 0 -> Bus 2 / Solar
    # gen 1 -> Bus 3 / Wind

    test_net.gen.at[0, "p_mw"] = solar_p
    test_net.gen.at[1, "p_mw"] = wind_p

    test_net.gen.at[0, "vm_pu"] = solar_support
    test_net.gen.at[1, "vm_pu"] = wind_support

    # Fixed inverter P-Q capability
    solar_qmax = np.sqrt(
        max(0.0, SOLAR_S_RATED**2 - solar_p**2)
    )

    wind_qmax = np.sqrt(
        max(0.0, WIND_S_RATED**2 - wind_p**2)
    )

    test_net.gen.at[0, "max_q_mvar"] = solar_qmax
    test_net.gen.at[0, "min_q_mvar"] = -solar_qmax

    test_net.gen.at[1, "max_q_mvar"] = wind_qmax
    test_net.gen.at[1, "min_q_mvar"] = -wind_qmax

    # -----------------------------
    # 3. Apply contingency
    # -----------------------------

    from_bus = int(row["From_Bus"])
    to_bus = int(row["To_Bus"])

    matching_lines = test_net.line[
        (
            (test_net.line["from_bus"] == from_bus - 1) &
            (test_net.line["to_bus"] == to_bus - 1)
        )
        |
        (
            (test_net.line["from_bus"] == to_bus - 1) &
            (test_net.line["to_bus"] == from_bus - 1)
        )
    ].index

    if len(matching_lines) == 0:
        validation_results.append({
            "Index": idx,
            "Solved": False,
            "Physics_Valid": False,
            "Reason": "Contingency line not found"
        })
        continue

    test_net.line.at[matching_lines[0], "in_service"] = False

    # -----------------------------
    # 4. Run actual power flow
    # -----------------------------

    try:

        pp.runpp(
            test_net,
            enforce_q_lims=True,
            numba=False
        )

        vmin = test_net.res_bus.vm_pu.min()
        vmax = test_net.res_bus.vm_pu.max()

        physics_valid = (
            vmin >= V_MIN - 1e-6 and
            vmax <= V_MAX + 1e-6
        )

        validation_results.append({
            "Index": idx,
            "Solved": True,
            "Physics_Valid": physics_valid,

            "AI_Solar_Headroom_MW": solar_hr,
            "AI_Wind_Headroom_MW": wind_hr,

            "AI_Solar_Support_pu": solar_support,
            "AI_Wind_Support_pu": wind_support,

            "AI_Total_Curtailment_MW":
                solar_hr + wind_hr,

            "Final_Min_Voltage": vmin,
            "Final_Max_Voltage": vmax,

            "Reason":
                "Secure" if physics_valid
                else "Voltage violation"
        })

    except Exception as e:

        validation_results.append({
            "Index": idx,
            "Solved": False,
            "Physics_Valid": False,
            "Reason": "Power flow failed"
        })


# ============================================================
# RESULTS
# ============================================================

physics_df = pd.DataFrame(validation_results)

valid_count = physics_df["Physics_Valid"].sum()
total_count = len(physics_df)

validation_rate = (
    100 * valid_count / total_count
)

print("=" * 60)
print("AI PHYSICS VALIDATION")
print("=" * 60)

print("Test scenarios:", total_count)

print(
    "Power-flow solved:",
    physics_df["Solved"].sum()
)

print(
    "Physics-valid:",
    f"{valid_count}/{total_count}"
)

print(
    "Physics validation rate:",
    round(validation_rate, 2),
    "%"
)

solved_df = physics_df[
    physics_df["Solved"] == True
]

if len(solved_df) > 0:

    print(
        "\nMinimum voltage after AI:",
        round(
            solved_df["Final_Min_Voltage"].min(),
            6
        )
    )

    print(
        "Maximum voltage after AI:",
        round(
            solved_df["Final_Max_Voltage"].max(),
            6
        )
    )

    print(
        "Average AI curtailment:",
        round(
            solved_df[
                "AI_Total_Curtailment_MW"
            ].mean(),
            4
        ),
        "MW"
    )

print("\nFailure reasons:")
print(physics_df["Reason"].value_counts())

physics_df.to_csv(
    "ai_physics_validation.csv",
    index=False
)

print(
    "\nSaved as: ai_physics_validation.csv"
)

AI PHYSICS VALIDATION
Test scenarios: 82
Power-flow solved: 82
Physics-valid: 37/82
Physics validation rate: 45.12 %

Minimum voltage after AI: 0.943374
Maximum voltage after AI: 1.04999
Average AI curtailment: 1.5041 MW

Failure reasons:
Reason
Voltage violation    45
Secure               37
Name: count, dtype: int64

Saved as: ai_physics_validation.csv


In [ ]:
# ============================================================
# ML STEP 8 — ANALYZE PHYSICS FAILURES
# ============================================================

failed = physics_df[
    (physics_df["Solved"] == True) &
    (physics_df["Physics_Valid"] == False)
].copy()

print("Failed cases:", len(failed))

print("\nFailed-case minimum voltage statistics:")
print(
    failed["Final_Min_Voltage"].describe()
)

print("\nHow close are failures to 0.95?")

for threshold in [0.949, 0.948, 0.945, 0.940]:
    count = (
        failed["Final_Min_Voltage"] >= threshold
    ).sum()

    print(
        f"Vmin >= {threshold}:",
        f"{count}/{len(failed)}",
        f"({100*count/len(failed):.1f}%)"
    )

print("\nWorst 10 failed cases:")

print(
    failed[
        [
            "Index",
            "Final_Min_Voltage",
            "Final_Max_Voltage",
            "AI_Solar_Headroom_MW",
            "AI_Wind_Headroom_MW",
            "AI_Solar_Support_pu",
            "AI_Wind_Support_pu"
        ]
    ]
    .sort_values("Final_Min_Voltage")
    .head(10)
    .to_string(index=False)
)

Failed cases: 45

Failed-case minimum voltage statistics:
count    45.000000
mean      0.949375
std       0.001087
min       0.943374
25%       0.949328
50%       0.949701
75%       0.949925
max       0.949999
Name: Final_Min_Voltage, dtype: float64

How close are failures to 0.95?
Vmin >= 0.949: 37/45 (82.2%)
Vmin >= 0.948: 42/45 (93.3%)
Vmin >= 0.945: 44/45 (97.8%)
Vmin >= 0.94: 45/45 (100.0%)

Worst 10 failed cases:
 Index  Final_Min_Voltage  Final_Max_Voltage  AI_Solar_Headroom_MW  AI_Wind_Headroom_MW  AI_Solar_Support_pu  AI_Wind_Support_pu
   409           0.943374           1.049763              7.095885             8.675202             1.049763            1.049660
   337           0.946997           1.039068              0.184367             0.439360             1.034633            1.039068
   322           0.947972           1.037723              0.045952             0.239993             1.025676            1.037723
   325           0.948584           1.048820              0.1

## 8. Physics-Based Safety Correction
Correct ML actions that do not satisfy the voltage criterion.


In [ ]:
# ============================================================
# ML STEP 9 — SAFETY CORRECTION LAYER
# Correct AI decisions using physics
# ============================================================

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import time

V_MIN = 0.95
V_MAX = 1.05

SOLAR_P_RATED = 163.0
WIND_P_RATED = 85.0

SOLAR_S_RATED = 1.10 * SOLAR_P_RATED
WIND_S_RATED = 1.10 * WIND_P_RATED


# ------------------------------------------------------------
# Physics evaluator
# ------------------------------------------------------------

def evaluate_ai_action(
    row,
    solar_hr,
    wind_hr,
    solar_support,
    wind_support
):

    test_net = pn.case9()

    # Load condition
    test_net.load["p_mw"] *= float(row["Load_Level"])
    test_net.load["q_mvar"] *= float(row["Load_Level"])

    # Keep actions inside physical limits
    solar_hr = np.clip(
        solar_hr, 0.0, float(row["Solar_Available_MW"])
    )

    wind_hr = np.clip(
        wind_hr, 0.0, float(row["Wind_Available_MW"])
    )

    solar_support = np.clip(solar_support, 1.00, 1.05)
    wind_support = np.clip(wind_support, 1.00, 1.05)

    # Active power after headroom
    solar_p = max(
        0.0,
        float(row["Solar_Available_MW"]) - solar_hr
    )

    wind_p = max(
        0.0,
        float(row["Wind_Available_MW"]) - wind_hr
    )

    test_net.gen.at[0, "p_mw"] = solar_p
    test_net.gen.at[1, "p_mw"] = wind_p

    test_net.gen.at[0, "vm_pu"] = solar_support
    test_net.gen.at[1, "vm_pu"] = wind_support

    # Fixed inverter apparent-power ratings
    solar_qmax = np.sqrt(
        max(0.0, SOLAR_S_RATED**2 - solar_p**2)
    )

    wind_qmax = np.sqrt(
        max(0.0, WIND_S_RATED**2 - wind_p**2)
    )

    test_net.gen.at[0, "max_q_mvar"] = solar_qmax
    test_net.gen.at[0, "min_q_mvar"] = -solar_qmax

    test_net.gen.at[1, "max_q_mvar"] = wind_qmax
    test_net.gen.at[1, "min_q_mvar"] = -wind_qmax

    # Apply contingency
    from_bus = int(row["From_Bus"])
    to_bus = int(row["To_Bus"])

    matching_lines = test_net.line[
        (
            (test_net.line["from_bus"] == from_bus - 1) &
            (test_net.line["to_bus"] == to_bus - 1)
        )
        |
        (
            (test_net.line["from_bus"] == to_bus - 1) &
            (test_net.line["to_bus"] == from_bus - 1)
        )
    ].index

    if len(matching_lines) == 0:
        return None

    test_net.line.at[
        matching_lines[0],
        "in_service"
    ] = False

    try:

        pp.runpp(
            test_net,
            enforce_q_lims=True,
            numba=False
        )

        vmin = float(test_net.res_bus.vm_pu.min())
        vmax = float(test_net.res_bus.vm_pu.max())

        secure = (
            vmin >= V_MIN - 1e-6 and
            vmax <= V_MAX + 1e-6
        )

        return {
            "secure": secure,
            "vmin": vmin,
            "vmax": vmax
        }

    except:
        return None


# ------------------------------------------------------------
# Safety correction
# ------------------------------------------------------------

def safety_correct(row, ai_action):

    base_solar_hr = max(
        0.0,
        float(ai_action["Solar_Headroom_MW"])
    )

    base_wind_hr = max(
        0.0,
        float(ai_action["Wind_Headroom_MW"])
    )

    base_solar_support = np.clip(
        float(ai_action["Solar_Support_pu"]),
        1.00,
        1.05
    )

    base_wind_support = np.clip(
        float(ai_action["Wind_Support_pu"]),
        1.00,
        1.05
    )

    # First: test raw AI decision
    raw = evaluate_ai_action(
        row,
        base_solar_hr,
        base_wind_hr,
        base_solar_support,
        base_wind_support
    )

    if raw is not None and raw["secure"]:

        return {
            "Safety_Result": "AI Direct",
            "Solar_Headroom_MW": base_solar_hr,
            "Wind_Headroom_MW": base_wind_hr,
            "Solar_Support_pu": base_solar_support,
            "Wind_Support_pu": base_wind_support,
            "Final_Min_Voltage": raw["vmin"],
            "Final_Max_Voltage": raw["vmax"],
            "Safety_Corrected": False
        }

    # --------------------------------------------------------
    # Small correction search around AI decision
    # --------------------------------------------------------

    support_steps = [
        0.001,
        0.002,
        0.003,
        0.005,
        0.010
    ]

    headroom_steps = [
        0.0,
        0.5,
        1.0,
        2.0,
        5.0
    ]

    candidates = []

    for support_step in support_steps:

        support_options = [
            (
                base_solar_support + support_step,
                base_wind_support
            ),
            (
                base_solar_support,
                base_wind_support + support_step
            ),
            (
                base_solar_support + support_step,
                base_wind_support + support_step
            )
        ]

        for solar_support, wind_support in support_options:

            for extra_hr in headroom_steps:

                headroom_options = [
                    (
                        base_solar_hr + extra_hr,
                        base_wind_hr
                    ),
                    (
                        base_solar_hr,
                        base_wind_hr + extra_hr
                    ),
                    (
                        base_solar_hr + extra_hr,
                        base_wind_hr + extra_hr
                    )
                ]

                for solar_hr, wind_hr in headroom_options:

                    result = evaluate_ai_action(
                        row,
                        solar_hr,
                        wind_hr,
                        solar_support,
                        wind_support
                    )

                    if result is None:
                        continue

                    if result["secure"]:

                        # Cost = extra curtailment first,
                        # then amount of support correction
                        extra_curtailment = (
                            max(0.0, solar_hr - base_solar_hr)
                            +
                            max(0.0, wind_hr - base_wind_hr)
                        )

                        support_change = (
                            abs(
                                solar_support
                                - base_solar_support
                            )
                            +
                            abs(
                                wind_support
                                - base_wind_support
                            )
                        )

                        candidates.append({
                            "Solar_Headroom_MW":
                                solar_hr,

                            "Wind_Headroom_MW":
                                wind_hr,

                            "Solar_Support_pu":
                                min(solar_support, 1.05),

                            "Wind_Support_pu":
                                min(wind_support, 1.05),

                            "Final_Min_Voltage":
                                result["vmin"],

                            "Final_Max_Voltage":
                                result["vmax"],

                            "Extra_Curtailment":
                                extra_curtailment,

                            "Support_Change":
                                support_change
                        })

    if len(candidates) == 0:

        return {
            "Safety_Result": "Fallback",
            "Solar_Headroom_MW": base_solar_hr,
            "Wind_Headroom_MW": base_wind_hr,
            "Solar_Support_pu": base_solar_support,
            "Wind_Support_pu": base_wind_support,
            "Final_Min_Voltage":
                raw["vmin"] if raw else np.nan,
            "Final_Max_Voltage":
                raw["vmax"] if raw else np.nan,
            "Safety_Corrected": False
        }

    # Prefer minimum extra curtailment,
    # then minimum support change
    candidates = sorted(
        candidates,
        key=lambda x: (
            x["Extra_Curtailment"],
            x["Support_Change"]
        )
    )

    best = candidates[0]

    best["Safety_Result"] = "Safety Corrected"
    best["Safety_Corrected"] = True

    return best


# ============================================================
# TEST SAFETY LAYER ON SAME 82 UNSEEN CASES
# ============================================================

safety_results = []

start = time.time()

for number, idx in enumerate(X_reg_test.index, start=1):

    row = reg_df.loc[idx]
    ai_action = ai_pred_df.loc[idx]

    result = safety_correct(
        row,
        ai_action
    )

    result["Index"] = idx
    safety_results.append(result)

    if number % 10 == 0:
        print(
            f"Completed {number}/{len(X_reg_test)}"
        )


safety_df = pd.DataFrame(safety_results)

elapsed = time.time() - start


# ============================================================
# FINAL SUMMARY
# ============================================================

direct = (
    safety_df["Safety_Result"] == "AI Direct"
).sum()

corrected = (
    safety_df["Safety_Result"] == "Safety Corrected"
).sum()

fallback = (
    safety_df["Safety_Result"] == "Fallback"
).sum()

secured = direct + corrected

print("\n" + "=" * 60)
print("AI + SAFETY LAYER RESULTS")
print("=" * 60)

print("Test scenarios:", len(safety_df))

print(
    "AI Direct Secure:",
    direct
)

print(
    "Safety Corrected:",
    corrected
)

print(
    "Fallback Required:",
    fallback
)

print(
    "\nDirect AI pass rate:",
    round(
        100 * direct / len(safety_df),
        2
    ),
    "%"
)

print(
    "AI + Safety secure rate:",
    round(
        100 * secured / len(safety_df),
        2
    ),
    "%"
)

print(
    "Fallback rate:",
    round(
        100 * fallback / len(safety_df),
        2
    ),
    "%"
)

secured_df = safety_df[
    safety_df["Safety_Result"] != "Fallback"
]

if len(secured_df) > 0:

    print(
        "\nMinimum secured voltage:",
        round(
            secured_df[
                "Final_Min_Voltage"
            ].min(),
            6
        )
    )

    print(
        "Maximum secured voltage:",
        round(
            secured_df[
                "Final_Max_Voltage"
            ].max(),
            6
        )
    )

print(
    "\nRuntime:",
    round(elapsed, 2),
    "seconds"
)

safety_df.to_csv(
    "ai_safety_validation.csv",
    index=False
)

print(
    "\nSaved as: ai_safety_validation.csv"
)

Completed 10/82
Completed 20/82
Completed 30/82
Completed 40/82
Completed 50/82
Completed 60/82
Completed 70/82
Completed 80/82

AI + SAFETY LAYER RESULTS
Test scenarios: 82
AI Direct Secure: 37
Safety Corrected: 45
Fallback Required: 0

Direct AI pass rate: 45.12 %
AI + Safety secure rate: 100.0 %
Fallback rate: 0.0 %

Minimum secured voltage: 0.95
Maximum secured voltage: 1.05

Runtime: 1229.4 seconds

Saved as: ai_safety_validation.csv


## 9. Operational Performance
Compare AI + Safety against optimizer targets.


In [ ]:
# ============================================================
# ML STEP 10 — OPERATIONAL PERFORMANCE
# AI + SAFETY vs OPTIMIZER
# ============================================================

import pandas as pd
import numpy as np

comparison_rows = []

for _, safety_row in safety_df.iterrows():

    idx = int(safety_row["Index"])

    # Original optimizer solution for the SAME test scenario
    true_row = reg_df.loc[idx]

    optimizer_curtailment = (
        float(true_row["Solar_Headroom_MW"]) +
        float(true_row["Wind_Headroom_MW"])
    )

    ai_safety_curtailment = (
        float(safety_row["Solar_Headroom_MW"]) +
        float(safety_row["Wind_Headroom_MW"])
    )

    comparison_rows.append({
        "Index": idx,

        "Optimizer_Curtailment_MW":
            optimizer_curtailment,

        "AI_Safety_Curtailment_MW":
            ai_safety_curtailment,

        "Curtailment_Difference_MW":
            ai_safety_curtailment -
            optimizer_curtailment,

        "Optimizer_Solar_Headroom_MW":
            float(true_row["Solar_Headroom_MW"]),

        "Optimizer_Wind_Headroom_MW":
            float(true_row["Wind_Headroom_MW"]),

        "AI_Safety_Solar_Headroom_MW":
            float(safety_row["Solar_Headroom_MW"]),

        "AI_Safety_Wind_Headroom_MW":
            float(safety_row["Wind_Headroom_MW"]),

        "Safety_Result":
            safety_row["Safety_Result"],

        "Final_Min_Voltage":
            safety_row["Final_Min_Voltage"],

        "Final_Max_Voltage":
            safety_row["Final_Max_Voltage"]
    })


comparison_df = pd.DataFrame(comparison_rows)

# ============================================================
# SUMMARY
# ============================================================

opt_avg = comparison_df[
    "Optimizer_Curtailment_MW"
].mean()

ai_avg = comparison_df[
    "AI_Safety_Curtailment_MW"
].mean()

extra_avg = comparison_df[
    "Curtailment_Difference_MW"
].mean()

zero_ai = (
    comparison_df["AI_Safety_Curtailment_MW"] <= 0.001
).sum()

zero_opt = (
    comparison_df["Optimizer_Curtailment_MW"] <= 0.001
).sum()

within_1mw = (
    comparison_df["Curtailment_Difference_MW"].abs()
    <= 1.0
).sum()

print("=" * 65)
print("OPERATIONAL PERFORMANCE")
print("=" * 65)

print("Test scenarios:", len(comparison_df))

print(
    "\nOptimizer average curtailment:",
    round(opt_avg, 4),
    "MW"
)

print(
    "AI + Safety average curtailment:",
    round(ai_avg, 4),
    "MW"
)

print(
    "Average difference:",
    round(extra_avg, 4),
    "MW"
)

print(
    "\nOptimizer zero-curtailment cases:",
    f"{zero_opt}/{len(comparison_df)}"
)

print(
    "AI + Safety zero-curtailment cases:",
    f"{zero_ai}/{len(comparison_df)}"
)

print(
    "\nAI + Safety within 1 MW of optimizer:",
    f"{within_1mw}/{len(comparison_df)}",
    f"({100*within_1mw/len(comparison_df):.2f}%)"
)

print(
    "\nMaximum AI + Safety curtailment:",
    round(
        comparison_df[
            "AI_Safety_Curtailment_MW"
        ].max(),
        4
    ),
    "MW"
)

print(
    "Maximum extra curtailment vs optimizer:",
    round(
        comparison_df[
            "Curtailment_Difference_MW"
        ].max(),
        4
    ),
    "MW"
)

# Headroom-required optimizer cases only
hr_cases = comparison_df[
    comparison_df["Optimizer_Curtailment_MW"] > 0.001
]

print("\n" + "-" * 65)
print("HEADROOM-REQUIRED CASES ONLY")
print("-" * 65)

print("Cases:", len(hr_cases))

if len(hr_cases) > 0:

    print(
        "Optimizer average:",
        round(
            hr_cases[
                "Optimizer_Curtailment_MW"
            ].mean(),
            4
        ),
        "MW"
    )

    print(
        "AI + Safety average:",
        round(
            hr_cases[
                "AI_Safety_Curtailment_MW"
            ].mean(),
            4
        ),
        "MW"
    )

# Save
comparison_df.to_csv(
    "ai_vs_optimizer_comparison.csv",
    index=False
)

print(
    "\nSaved as: ai_vs_optimizer_comparison.csv"
)

OPERATIONAL PERFORMANCE
Test scenarios: 82

Optimizer average curtailment: 1.256 MW
AI + Safety average curtailment: 1.6321 MW
Average difference: 0.3762 MW

Optimizer zero-curtailment cases: 74/82
AI + Safety zero-curtailment cases: 55/82

AI + Safety within 1 MW of optimizer: 74/82 (90.24%)

Maximum AI + Safety curtailment: 27.2349 MW
Maximum extra curtailment vs optimizer: 7.3897 MW

-----------------------------------------------------------------
HEADROOM-REQUIRED CASES ONLY
-----------------------------------------------------------------
Cases: 8
Optimizer average: 12.8736 MW
AI + Safety average: 14.913 MW

Saved as: ai_vs_optimizer_comparison.csv


## 10. Fixed-Headroom Study Baseline
Compare dynamic EZHAL actions against fixed 5/10/15/20% headroom study baselines.


In [ ]:
# ============================================================
# STEP 11 — FIXED HEADROOM BASELINE
# Compare Fixed 5/10/15/20% vs Dynamic AI + Safety
# ============================================================

import pandas as pd
import numpy as np
import pandapower as pp
import pandapower.networks as pn

fixed_levels = [0.05, 0.10, 0.15, 0.20]

baseline_results = []

for fixed_hr in fixed_levels:

    secure_count = 0
    total_curtailment = []
    solved_count = 0

    for idx in X_reg_test.index:

        row = reg_df.loc[idx]

        # Fixed percentage of AVAILABLE renewable power
        solar_hr = (
            float(row["Solar_Available_MW"]) * fixed_hr
        )

        wind_hr = (
            float(row["Wind_Available_MW"]) * fixed_hr
        )

        # Give both resources maximum voltage-support setpoint
        solar_support = 1.05
        wind_support = 1.05

        result = evaluate_ai_action(
            row,
            solar_hr,
            wind_hr,
            solar_support,
            wind_support
        )

        curtailment = solar_hr + wind_hr
        total_curtailment.append(curtailment)

        if result is not None:

            solved_count += 1

            if result["secure"]:
                secure_count += 1

        baseline_results.append({
            "Index": idx,
            "Fixed_Headroom_Percent":
                int(fixed_hr * 100),

            "Curtailment_MW":
                curtailment,

            "Solved":
                result is not None,

            "Secure":
                (
                    result["secure"]
                    if result is not None
                    else False
                ),

            "Final_Min_Voltage":
                (
                    result["vmin"]
                    if result is not None
                    else np.nan
                ),

            "Final_Max_Voltage":
                (
                    result["vmax"]
                    if result is not None
                    else np.nan
                )
        })

    print("\n" + "=" * 55)
    print(f"FIXED HEADROOM: {int(fixed_hr*100)}%")
    print("=" * 55)

    print(
        "Secure:",
        f"{secure_count}/{len(X_reg_test)}",
        f"({100*secure_count/len(X_reg_test):.2f}%)"
    )

    print(
        "Average curtailment:",
        round(np.mean(total_curtailment), 4),
        "MW"
    )


baseline_df = pd.DataFrame(baseline_results)

baseline_df.to_csv(
    "fixed_headroom_baseline.csv",
    index=False
)

print("\n" + "=" * 60)
print("DYNAMIC AI + SAFETY")
print("=" * 60)

print(
    "Secure:",
    f"{len(safety_df)}/{len(safety_df)}"
)

print(
    "Average curtailment:",
    round(
        comparison_df[
            "AI_Safety_Curtailment_MW"
        ].mean(),
        4
    ),
    "MW"
)

print(
    "\nSaved as: fixed_headroom_baseline.csv"
)


FIXED HEADROOM: 5%
Secure: 72/82 (87.80%)
Average curtailment: 9.5061 MW

FIXED HEADROOM: 10%
Secure: 72/82 (87.80%)
Average curtailment: 19.0122 MW

FIXED HEADROOM: 15%
Secure: 71/82 (86.59%)
Average curtailment: 28.5183 MW

FIXED HEADROOM: 20%
Secure: 71/82 (86.59%)
Average curtailment: 38.0244 MW

DYNAMIC AI + SAFETY
Secure: 82/82
Average curtailment: 1.6321 MW

Saved as: fixed_headroom_baseline.csv


## 11. Demo Scenario
Select a real unseen test scenario for the dashboard/presentation.


In [ ]:
# ============================================================
# STEP 12 — SELECT A REAL DEMO SCENARIO
# For presentation + dashboard
# ============================================================

demo_candidates = []

for _, s in safety_df.iterrows():

    idx = int(s["Index"])
    original = reg_df.loc[idx]

    # Prefer an interesting case that required headroom
    total_hr = (
        float(s["Solar_Headroom_MW"]) +
        float(s["Wind_Headroom_MW"])
    )

    demo_candidates.append({
        "Index": idx,

        "Load_Level":
            float(original["Load_Level"]),

        "Solar_Level":
            float(original["Solar_Level"]),

        "Wind_Level":
            float(original["Wind_Level"]),

        "From_Bus":
            int(original["From_Bus"]),

        "To_Bus":
            int(original["To_Bus"]),

        "Before_Min_Voltage":
            float(original["Base_Min_Voltage"]),

        "Solar_Headroom_MW":
            float(s["Solar_Headroom_MW"]),

        "Wind_Headroom_MW":
            float(s["Wind_Headroom_MW"]),

        "Solar_Support_pu":
            float(s["Solar_Support_pu"]),

        "Wind_Support_pu":
            float(s["Wind_Support_pu"]),

        "Total_Curtailment_MW":
            total_hr,

        "After_Min_Voltage":
            float(s["Final_Min_Voltage"]),

        "After_Max_Voltage":
            float(s["Final_Max_Voltage"]),

        "Safety_Result":
            s["Safety_Result"]
    })


demo_df = pd.DataFrame(demo_candidates)

# Choose a case with:
# 1. actual headroom
# 2. weak initial voltage
# 3. secure final voltage
interesting = demo_df[
    (demo_df["Total_Curtailment_MW"] > 0.5) &
    (demo_df["Before_Min_Voltage"] < 0.95) &
    (demo_df["After_Min_Voltage"] >= 0.95) &
    (demo_df["After_Max_Voltage"] <= 1.05)
].copy()

if len(interesting) > 0:

    # Prefer a clear improvement without extreme curtailment
    interesting["Voltage_Improvement"] = (
        interesting["After_Min_Voltage"] -
        interesting["Before_Min_Voltage"]
    )

    demo_case = interesting.sort_values(
        ["Voltage_Improvement", "Total_Curtailment_MW"],
        ascending=[False, True]
    ).iloc[0]

else:

    demo_case = demo_df.sort_values(
        "Before_Min_Voltage"
    ).iloc[0]


print("=" * 60)
print("SELECTED REAL DEMO SCENARIO")
print("=" * 60)

print("Scenario index:", int(demo_case["Index"]))

print(
    "\nOperating condition:"
)

print(
    "Load:",
    round(demo_case["Load_Level"] * 100),
    "%"
)

print(
    "Solar availability:",
    round(demo_case["Solar_Level"] * 100),
    "%"
)

print(
    "Wind availability:",
    round(demo_case["Wind_Level"] * 100),
    "%"
)

print(
    "Contingency:",
    f'Line {int(demo_case["From_Bus"])}-{int(demo_case["To_Bus"])} outage'
)

print("\n--- BEFORE ---")

print(
    "Minimum voltage:",
    round(demo_case["Before_Min_Voltage"], 4),
    "pu"
)

print("\n--- AI + SAFETY DECISION ---")

print(
    "Solar support:",
    round(demo_case["Solar_Support_pu"], 4),
    "pu"
)

print(
    "Wind support:",
    round(demo_case["Wind_Support_pu"], 4),
    "pu"
)

print(
    "Solar headroom:",
    round(demo_case["Solar_Headroom_MW"], 2),
    "MW"
)

print(
    "Wind headroom:",
    round(demo_case["Wind_Headroom_MW"], 2),
    "MW"
)

print(
    "Total curtailment:",
    round(demo_case["Total_Curtailment_MW"], 2),
    "MW"
)

print(
    "Decision:",
    demo_case["Safety_Result"]
)

print("\n--- AFTER ---")

print(
    "Minimum voltage:",
    round(demo_case["After_Min_Voltage"], 4),
    "pu"
)

print(
    "Maximum voltage:",
    round(demo_case["After_Max_Voltage"], 4),
    "pu"
)

print(
    "Grid status:",
    "SECURE"
)

demo_df.to_csv(
    "demo_scenarios.csv",
    index=False
)

print("\nSaved as: demo_scenarios.csv")

SELECTED REAL DEMO SCENARIO
Scenario index: 313

Operating condition:
Load: 115 %
Solar availability: 70 %
Wind availability: 100 %
Contingency: Line 7-8 outage

--- BEFORE ---
Minimum voltage: 0.8399 pu

--- AI + SAFETY DECISION ---
Solar support: 1.05 pu
Wind support: 1.0497 pu
Solar headroom: 0.12 MW
Wind headroom: 10.07 MW
Total curtailment: 10.2 MW
Decision: AI Direct

--- AFTER ---
Minimum voltage: 0.9514 pu
Maximum voltage: 1.05 pu
Grid status: SECURE

Saved as: demo_scenarios.csv


In [ ]:
# ============================================================
# STEP 13 — DEMO SCENARIO VS 5% FIXED BASELINE
# ============================================================

demo_idx = 313
row = reg_df.loc[demo_idx]

# Fixed 5% headroom
fixed_solar_hr = float(row["Solar_Available_MW"]) * 0.05
fixed_wind_hr  = float(row["Wind_Available_MW"]) * 0.05

fixed_result = evaluate_ai_action(
    row,
    fixed_solar_hr,
    fixed_wind_hr,
    1.05,
    1.05
)

fixed_total_hr = fixed_solar_hr + fixed_wind_hr

# Our dynamic AI + Safety result
our_row = safety_df[
    safety_df["Index"] == demo_idx
].iloc[0]

our_total_hr = (
    float(our_row["Solar_Headroom_MW"]) +
    float(our_row["Wind_Headroom_MW"])
)

reduction_percent = (
    (fixed_total_hr - our_total_hr)
    / fixed_total_hr
) * 100

print("=" * 60)
print("DEMO SCENARIO — DYNAMIC vs FIXED 5%")
print("=" * 60)

print("\nFIXED 5%:")
print("Curtailment:", round(fixed_total_hr, 2), "MW")

if fixed_result is not None:
    print("Minimum voltage:", round(fixed_result["vmin"], 4), "pu")
    print("Maximum voltage:", round(fixed_result["vmax"], 4), "pu")
    print(
        "Status:",
        "SECURE" if fixed_result["secure"] else "INSECURE"
    )

print("\nOUR DYNAMIC SYSTEM:")
print("Curtailment:", round(our_total_hr, 2), "MW")
print(
    "Minimum voltage:",
    round(float(our_row["Final_Min_Voltage"]), 4),
    "pu"
)
print(
    "Maximum voltage:",
    round(float(our_row["Final_Max_Voltage"]), 4),
    "pu"
)
print("Status: SECURE")

print(
    "\nCurtailment reduction vs fixed 5%:",
    round(reduction_percent, 2),
    "%"
)

DEMO SCENARIO — DYNAMIC vs FIXED 5%

FIXED 5%:
Curtailment: 9.96 MW
Minimum voltage: 0.9152 pu
Maximum voltage: 1.05 pu
Status: INSECURE

OUR DYNAMIC SYSTEM:
Curtailment: 10.2 MW
Minimum voltage: 0.9514 pu
Maximum voltage: 1.05 pu
Status: SECURE

Curtailment reduction vs fixed 5%: -2.42 %


## 12. Final Prototype KPIs


In [ ]:
# ============================================================
# STEP 14 — FINAL PROJECT KPI SUMMARY
# ============================================================

print("=" * 65)
print("EZHAL — FINAL PROTOTYPE RESULTS")
print("=" * 65)

# 1. Simulation
print("\n[1] SIMULATION")
print("Total simulated scenarios: 875")
print("Initially insecure scenarios: 659")

# 2. Optimizer
print("\n[2] OPTIMIZATION")
print("Feasible corrective solutions: 434 / 659")
print("Support-only solutions: 378")
print("Headroom + support solutions: 56")
print("Optimizer physics validation: 434 / 434 (100%)")

# 3. ML classifier
print("\n[3] MACHINE LEARNING — ACTION CLASSIFICATION")
print("Clean ML scenarios: 654")
print("Input features: 16")
print("Best model: Extra Trees")
print("Accuracy: 97.62%")
print("Macro F1: 95.23%")
print("Balanced Accuracy: 98.65%")

# 4. ML regression
print("\n[4] MACHINE LEARNING — WHERE + HOW MUCH")
print("Best model: Extra Trees")
print("Average R2: 87.38%")
print("Solar support R2: 98.51%")
print("Wind support R2: 99.24%")
print("Solar headroom R2: 76.99%")
print("Wind headroom R2: 74.79%")

# 5. Physics validation
print("\n[5] UNSEEN PHYSICS TEST")
print("Unseen feasible test scenarios: 82")
print("Direct AI secure: 37 / 82 (45.12%)")
print("AI + Safety secured: 82 / 82")
print("Final voltage criterion: 0.95–1.05 pu")

# 6. Operational comparison
print("\n[6] DYNAMIC vs FIXED 5% BASELINE")
print("Fixed 5% average curtailment: 9.5061 MW")
print("Dynamic AI + Safety average curtailment: 1.6321 MW")
print("Curtailment reduction: 82.83%")
print("Fixed 5% secure cases: 72 / 82 (87.80%)")
print("Dynamic AI + Safety secure cases: 82 / 82")

# 7. Demo scenario
print("\n[7] REAL DEMO TEST SCENARIO")
print("Scenario: Line 7-8 outage")
print("Load: 115%")
print("Solar availability: 70%")
print("Wind availability: 100%")
print("Voltage before: 0.8399 pu")
print("Voltage after: 0.9514 pu")
print("Solar headroom: 0.12 MW")
print("Wind headroom: 10.07 MW")
print("Total dynamic curtailment: 10.20 MW")
print("Fixed 5% voltage: 0.9152 pu — INSECURE")
print("Dynamic system voltage: 0.9514 pu — SECURE")

print("\n" + "=" * 65)
print("SUMMARY COMPLETE")
print("=" * 65)

EZHAL — FINAL PROTOTYPE RESULTS

[1] SIMULATION
Total simulated scenarios: 875
Initially insecure scenarios: 659

[2] OPTIMIZATION
Feasible corrective solutions: 434 / 659
Support-only solutions: 378
Headroom + support solutions: 56
Optimizer physics validation: 434 / 434 (100%)

[3] MACHINE LEARNING — ACTION CLASSIFICATION
Clean ML scenarios: 654
Input features: 16
Best model: Extra Trees
Accuracy: 97.62%
Macro F1: 95.23%
Balanced Accuracy: 98.65%

[4] MACHINE LEARNING — WHERE + HOW MUCH
Best model: Extra Trees
Average R2: 87.38%
Solar support R2: 98.51%
Wind support R2: 99.24%
Solar headroom R2: 76.99%
Wind headroom R2: 74.79%

[5] UNSEEN PHYSICS TEST
Unseen feasible test scenarios: 82
Direct AI secure: 37 / 82 (45.12%)
AI + Safety secured: 82 / 82
Final voltage criterion: 0.95–1.05 pu

[6] DYNAMIC vs FIXED 5% BASELINE
Fixed 5% average curtailment: 9.5061 MW
Dynamic AI + Safety average curtailment: 1.6321 MW
Curtailment reduction: 82.83%
Fixed 5% secure cases: 72 / 82 (87.80%)
Dynami

## 13. Rebuild the Final Feasible Regression Dataset
Reconstruct the exact 16-feature dataset used by the live prototype.


In [ ]:
import pandas as pd
import numpy as np

# Load saved optimizer dataset
df = pd.read_csv("final_v2_optimizer_dataset.csv")

# The 16 PRE-ACTION features used by EZHAL ML
features = [
    "Load_Level",
    "Solar_Level",
    "Wind_Level",
    "Solar_Available_MW",
    "Wind_Available_MW",
    "From_Bus",
    "To_Bus",
    "Base_Min_Voltage",
    "Base_Max_Voltage",
    "Weakest_Bus_Before",
    "Base_Solar_Q_MVAr",
    "Base_Wind_Q_MVAr",
    "Base_Solar_Q_Capability_MVAr",
    "Base_Wind_Q_Capability_MVAr",
    "Base_Solar_Q_Margin_MVAr",
    "Base_Wind_Q_Margin_MVAr"
]

targets = [
    "Solar_Headroom_MW",
    "Wind_Headroom_MW",
    "Solar_Support_pu",
    "Wind_Support_pu"
]

# Remove rows missing ML input features
clean_df = df.dropna(subset=features).copy()

# Regression dataset = only feasible scenarios with optimizer actions
reg_df = clean_df[
    (clean_df["Status"] == "Feasible") &
    clean_df[targets].notna().all(axis=1)
].copy()

# IMPORTANT: recreate the index used during the old ML test
reg_df = reg_df.reset_index(drop=True)

print("Clean ML rows:", len(clean_df))
print("Regression feasible rows:", len(reg_df))

print("\nExpected:")
print("Clean ML rows = 654")
print("Regression feasible rows = 429")

print("\nSafety index range:")
print(safety["Index"].min(), "to", safety["Index"].max())

print("\nExample scenario linked to Safety Index 7:")
display(reg_df.loc[[7], features + ["Action_Type"]])

Clean ML rows: 654
Regression feasible rows: 429

Expected:
Clean ML rows = 654
Regression feasible rows = 429

Safety index range:
7 to 409

Example scenario linked to Safety Index 7:


,Load_Level,Solar_Level,Wind_Level,Solar_Available_MW,Wind_Available_MW,From_Bus,To_Bus,Base_Min_Voltage,Base_Max_Voltage,Weakest_Bus_Before,Base_Solar_Q_MVAr,Base_Wind_Q_MVAr,Base_Solar_Q_Capability_MVAr,Base_Wind_Q_Capability_MVAr,Base_Solar_Q_Margin_MVAr,Base_Wind_Q_Margin_MVAr,Action_Type
7,0.9,0.6,1.0,97.8,85.0,5,6,0.93997,1.0,5.0,10.77263,2.167078,150.278575,38.951893,139.505945,36.784816,Support Only


## 14. Live Multi-Criteria Physics Check
Validate voltage security, reactive-power capability, and inverter apparent-power capacity.


In [ ]:
import numpy as np
import pandapower as pp
import pandapower.networks as pn

V_MIN = 0.95
V_MAX = 1.05

SOLAR_S_RATED = 179.3
WIND_S_RATED = 93.5


def live_security_check(
    row,
    solar_hr,
    wind_hr,
    solar_support,
    wind_support
):

    # -----------------------------------------
    # 1. Build the same case9 grid
    # -----------------------------------------
    test_net = pn.case9()

    test_net.load["p_mw"] *= float(row["Load_Level"])
    test_net.load["q_mvar"] *= float(row["Load_Level"])

    solar_available = float(row["Solar_Available_MW"])
    wind_available = float(row["Wind_Available_MW"])

    # Keep AI actions inside physical bounds
    solar_hr = np.clip(solar_hr, 0.0, solar_available)
    wind_hr = np.clip(wind_hr, 0.0, wind_available)

    solar_support = np.clip(solar_support, 1.00, 1.05)
    wind_support = np.clip(wind_support, 1.00, 1.05)

    # -----------------------------------------
    # 2. Final active power
    # -----------------------------------------
    solar_p = solar_available - solar_hr
    wind_p = wind_available - wind_hr

    test_net.gen.at[0, "p_mw"] = solar_p
    test_net.gen.at[1, "p_mw"] = wind_p

    test_net.gen.at[0, "vm_pu"] = solar_support
    test_net.gen.at[1, "vm_pu"] = wind_support

    # -----------------------------------------
    # 3. Reactive capability
    # -----------------------------------------
    solar_qmax = np.sqrt(
        max(0.0, SOLAR_S_RATED**2 - solar_p**2)
    )

    wind_qmax = np.sqrt(
        max(0.0, WIND_S_RATED**2 - wind_p**2)
    )

    test_net.gen.at[0, "max_q_mvar"] = solar_qmax
    test_net.gen.at[0, "min_q_mvar"] = -solar_qmax

    test_net.gen.at[1, "max_q_mvar"] = wind_qmax
    test_net.gen.at[1, "min_q_mvar"] = -wind_qmax

    # -----------------------------------------
    # 4. Apply contingency
    # -----------------------------------------
    from_bus = int(row["From_Bus"])
    to_bus = int(row["To_Bus"])

    matching = test_net.line[
        (
            (test_net.line["from_bus"] == from_bus - 1) &
            (test_net.line["to_bus"] == to_bus - 1)
        )
        |
        (
            (test_net.line["from_bus"] == to_bus - 1) &
            (test_net.line["to_bus"] == from_bus - 1)
        )
    ].index

    if len(matching) == 0:
        return {"Solved": False}

    test_net.line.at[matching[0], "in_service"] = False

    # -----------------------------------------
    # 5. LIVE POWER FLOW
    # -----------------------------------------
    try:

        pp.runpp(
            test_net,
            enforce_q_lims=True,
            numba=False
        )

    except:
        return {"Solved": False}

    # -----------------------------------------
    # 6. Actual physics results
    # -----------------------------------------
    vmin = float(test_net.res_bus.vm_pu.min())
    vmax = float(test_net.res_bus.vm_pu.max())

    solar_q = float(test_net.res_gen.at[0, "q_mvar"])
    wind_q = float(test_net.res_gen.at[1, "q_mvar"])

    solar_used_s = np.sqrt(
        solar_p**2 + solar_q**2
    )

    wind_used_s = np.sqrt(
        wind_p**2 + wind_q**2
    )

    # -----------------------------------------
    # CHECK 1 — VOLTAGE
    # -----------------------------------------
    voltage_pass = (
        vmin >= V_MIN - 1e-6
        and vmax <= V_MAX + 1e-6
    )

    # -----------------------------------------
    # CHECK 2 — REACTIVE POWER
    # -----------------------------------------
    q_pass = (
        abs(solar_q) <= solar_qmax + 1e-6
        and abs(wind_q) <= wind_qmax + 1e-6
    )

    # -----------------------------------------
    # CHECK 3 — INVERTER CAPACITY
    # -----------------------------------------
    inverter_pass = (
        solar_used_s <= SOLAR_S_RATED + 1e-6
        and wind_used_s <= WIND_S_RATED + 1e-6
    )

    overall_pass = (
        voltage_pass
        and q_pass
        and inverter_pass
    )

    return {
        "Solved": True,

        "Vmin": vmin,
        "Vmax": vmax,

        "Solar_Q": solar_q,
        "Solar_Qmax": solar_qmax,

        "Wind_Q": wind_q,
        "Wind_Qmax": wind_qmax,

        "Solar_Used_S": solar_used_s,
        "Wind_Used_S": wind_used_s,

        "Voltage_PASS": voltage_pass,
        "Q_PASS": q_pass,
        "Inverter_PASS": inverter_pass,

        "Overall_PASS": overall_pass
    }


print("✅ EZHAL Live Multi-Criteria Physics Check ready")

✅ EZHAL Live Multi-Criteria Physics Check ready


## 15. End-to-End Live ML + Physics Demo
Scenario → trained ML → action → live physics validation.


In [ ]:
# ============================================================
# EZHAL — END-TO-END LIVE ML + PHYSICS TEST
# Scenario -> Trained ML -> AI Decision -> Physics -> 3 Checks
# ============================================================

import pandas as pd
import numpy as np
import joblib

# ------------------------------------------------------------
# 1. LOAD TRAINED ML MODELS
# ------------------------------------------------------------

classifier = joblib.load("final_action_classifier.pkl")
regressor  = joblib.load("final_action_regressor.pkl")

# ------------------------------------------------------------
# 2. LOAD SCENARIO DATA
# ------------------------------------------------------------

reg_df = pd.read_csv("regression_feasible_clean.csv")

features = [
    "Load_Level",
    "Solar_Level",
    "Wind_Level",
    "Solar_Available_MW",
    "Wind_Available_MW",
    "From_Bus",
    "To_Bus",
    "Base_Min_Voltage",
    "Base_Max_Voltage",
    "Weakest_Bus_Before",
    "Base_Solar_Q_MVAr",
    "Base_Wind_Q_MVAr",
    "Base_Solar_Q_Capability_MVAr",
    "Base_Wind_Q_Capability_MVAr",
    "Base_Solar_Q_Margin_MVAr",
    "Base_Wind_Q_Margin_MVAr"
]

demo_idx = 313

row = reg_df[
    reg_df["Index"] == demo_idx
].iloc[0]

X_live = pd.DataFrame(
    [row[features].values],
    columns=features
).astype(float)

# ------------------------------------------------------------
# 3. LIVE ML CLASSIFICATION
# What type of action is needed?
# ------------------------------------------------------------

predicted_action = classifier.predict(X_live)[0]

# ------------------------------------------------------------
# 4. LIVE ML REGRESSION
# Where + how much?
# ------------------------------------------------------------

prediction = regressor.predict(X_live)[0]

solar_hr = max(
    0.0,
    min(float(prediction[0]), float(row["Solar_Available_MW"]))
)

wind_hr = max(
    0.0,
    min(float(prediction[1]), float(row["Wind_Available_MW"]))
)

solar_support = float(
    np.clip(prediction[2], 1.00, 1.05)
)

wind_support = float(
    np.clip(prediction[3], 1.00, 1.05)
)

# ------------------------------------------------------------
# 5. SHOW LIVE ML DECISION
# ------------------------------------------------------------

print("=" * 65)
print("EZHAL — LIVE MACHINE LEARNING DECISION")
print("=" * 65)

print("\nScenario:", demo_idx)

print(
    "Grid condition:",
    f"Load {row['Load_Level']*100:.0f}% |",
    f"Solar {row['Solar_Level']*100:.0f}% |",
    f"Wind {row['Wind_Level']*100:.0f}% |",
    f"Line {int(row['From_Bus'])}-{int(row['To_Bus'])}"
)

print("\n🤖 ML CLASSIFIER")
print("Predicted Action:", predicted_action)

print("\n🤖 ML REGRESSOR — WHERE + HOW MUCH")

print(
    "Solar Headroom:",
    round(solar_hr, 3),
    "MW"
)

print(
    "Wind Headroom:",
    round(wind_hr, 3),
    "MW"
)

print(
    "Solar Support:",
    round(solar_support, 4),
    "pu"
)

print(
    "Wind Support:",
    round(wind_support, 4),
    "pu"
)

# ------------------------------------------------------------
# 6. LIVE PHYSICS CHECK
# ------------------------------------------------------------

physics = live_security_check(
    row=row,
    solar_hr=solar_hr,
    wind_hr=wind_hr,
    solar_support=solar_support,
    wind_support=wind_support
)

print("\n" + "=" * 65)
print("LIVE PHYSICS VALIDATION")
print("=" * 65)

if not physics["Solved"]:

    print("\n❌ POWER FLOW FAILED")

else:

    print("\n1️⃣ VOLTAGE SECURITY")
    print(
        "Range:",
        f"{physics['Vmin']:.4f}",
        "→",
        f"{physics['Vmax']:.4f}",
        "pu"
    )
    print(
        "Result:",
        "✅ PASS"
        if physics["Voltage_PASS"]
        else "❌ FAIL"
    )

    print("\n2️⃣ REACTIVE POWER CAPABILITY")

    print(
        "Solar:",
        f"{physics['Solar_Q']:.2f}",
        "/",
        f"{physics['Solar_Qmax']:.2f}",
        "MVAr"
    )

    print(
        "Wind:",
        f"{physics['Wind_Q']:.2f}",
        "/",
        f"{physics['Wind_Qmax']:.2f}",
        "MVAr"
    )

    print(
        "Result:",
        "✅ PASS"
        if physics["Q_PASS"]
        else "❌ FAIL"
    )

    print("\n3️⃣ INVERTER CAPACITY")

    print(
        "Solar:",
        f"{physics['Solar_Used_S']:.2f}",
        "/ 179.30 MVA"
    )

    print(
        "Wind:",
        f"{physics['Wind_Used_S']:.2f}",
        "/ 93.50 MVA"
    )

    print(
        "Result:",
        "✅ PASS"
        if physics["Inverter_PASS"]
        else "❌ FAIL"
    )

    print("\n" + "=" * 65)

    if physics["Overall_PASS"]:
        print("🟢 LIVE ML DECISION: PHYSICALLY SECURE")
    else:
        print("🔴 LIVE ML DECISION: SAFETY CORRECTION REQUIRED")

    print("=" * 65)

EZHAL — LIVE MACHINE LEARNING DECISION

Scenario: 313
Grid condition: Load 115% | Solar 70% | Wind 100% | Line 7-8

🤖 ML CLASSIFIER
Predicted Action: Headroom + Support

🤖 ML REGRESSOR — WHERE + HOW MUCH
Solar Headroom: 0.121 MW
Wind Headroom: 10.075 MW
Solar Support: 1.05 pu
Wind Support: 1.0497 pu

LIVE PHYSICS VALIDATION

1️⃣ VOLTAGE SECURITY
Range: 0.9514 → 1.0500 pu
Result: ✅ PASS

2️⃣ REACTIVE POWER CAPABILITY
Solar: 25.06 / 138.41 MVAr
Wind: 55.93 / 55.93 MVAr
Result: ✅ PASS

3️⃣ INVERTER CAPACITY
Solar: 116.70 / 179.30 MVA
Wind: 93.50 / 93.50 MVA
Result: ✅ PASS

🟢 LIVE ML DECISION: PHYSICALLY SECURE


## Notes on Reproducibility

Some cells are intentionally computationally expensive:

- The full corrected simulation generates **875 scenarios**.
- The final optimizer processes the insecure scenarios and can take tens of minutes.
- The physics-based safety correction performs repeated power-flow evaluations and is also slow.

For hackathon review, the saved outputs/models can be used directly by the Streamlit dashboard instead of rerunning every expensive training/optimization stage.

### Key reported prototype results
- **875** total simulated scenarios
- **659** initially insecure scenarios
- **95.23%** classifier Macro F1
- **82/82** unseen feasible test scenarios secured after AI + physics-based safety correction
- **82.83%** lower average curtailment versus the 5% fixed-headroom study baseline

These are prototype-study results, not claims of real-grid deployment performance.
